In [2]:
%matplotlib inline
from beamforming import *
from plot_utils import *
from util import *
from sensor_array import *
from psd_estimation import *
from ml_util import *

import os
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from ipywidgets import VBox, HBox, FloatSlider, Output, Label, Dropdown, FloatLogSlider, Text, Button, Layout, IntSlider
from IPython.display import display, clear_output, Audio
import json
import pathlib

In [4]:
fig_width = 11
sr = 44100
c = 344
duration = 5
N = WINDOW_LENGTH
step = WINDOW_STEP
window = WINDOW
beamforming_resolution = 100
fmin = FMIN
fmax = FMAX
VN, f, validf = NSTFT, VALID_FREQUENCIES, VALID_FREQUENCY_INDICES

path = "../dataset/synthetic_training/neo_7_mvdr"
droneType, distance, algo, featureType = str(pathlib.Path(path+"_stft").name).split("_")

with open(path+"_mfcc.npy", 'rb') as fil:
    mfcc_feature = np.load(fil)
with open(path+"_stft.npy", 'rb') as fil:
    stft_feature = np.load(fil)

size = len(mfcc_feature)
N = FEATURE_LENGTH

w_start = IntSlider(
    value=0, min=0, max=max(size-N, 0),
    step=1,
    description="Pos", continuous_update=True
)
w_start.layout.width = f"{fig_width * 80 * 1.15}px"

out = Output()

class Sampler:
    start = 0
    end = duration
    s_idx = 0
    e_idx = sr * duration

sampler = Sampler()

def update_plot(change):

    with out:
        clear_output(wait=True)

        fig = plt.figure(figsize=(fig_width*1.2, fig_width * 0.3))
        ax_mfcc = fig.add_subplot(121)
        ax_stft = fig.add_subplot(122)

        s_idx = w_start.value
        e_idx = min(s_idx + N, size)
        s_idx = max(0, e_idx - N)
        NN = e_idx - s_idx

        plot_mfcc_feature(ax_mfcc, np.arange(NN), normalize_raw_mfcc(mfcc_feature[s_idx:e_idx].T))
        plot_stft_feature(ax_stft, f, np.arange(NN), normalize_raw_stft(stft_feature[s_idx:e_idx].T))

        plt.show()

# Hook up events and render once
for w in (w_start,):
    w.observe(update_plot, names="value")

display(VBox([
    w_start,
    out
]))

update_plot(None)